In [ ]:
import yfinance as yf
import pandas as pd

tickers = [
    "BTC-USD", "ETH-USD", "SOL-USD",
    "NVDA", "AAPL", "MSFT", "GOOGL", "QQQ"
]

# 1. Baixar Dados
dados = yf.download(tickers, period="730d", interval="1h", progress=False)

# 2. Pegar apenas o Preço de Fechamento ('Close')
df = dados['Close']

df_alinhado = df.resample('1h').last().ffill()

df_final = df_alinhado.dropna()

if len(df_final) > 0:
    nome_arquivo = "dataset_principal.csv"
    df_final.to_csv(nome_arquivo)
    print(f"\nDataset salvo: {nome_arquivo}")
    print("Amostra:")
    print(df_final.tail())
else:
    print("O dataset ainda está vazio. Tente aumentar o 'period' ou verifique sua conexão.")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib
import gc

INPUT_FILE = "dataset_principal.csv"
WINDOW_SIZE = 168       # 1 semana de histórico (Input)
FORECAST_HORIZON = 24   # Previsão 24h à frente
TARGET_ASSET = 'BTC-USD'

try:
    df = pd.read_csv(INPUT_FILE, index_col=0)
    df.index = pd.to_datetime(df.index)
except FileNotFoundError:
    print(f"Arquivo '{INPUT_FILE}' não encontrado.")
    exit()

# A. Retornos Logarítmicos
df_log = np.log(df / df.shift(1))
df_log = df_log.add_suffix('_LogRet')

# B. Volatilidade Recente (Contexto)
df_vol_recent = df_log.rolling(window=24).std().add_suffix('_Vol24h')

# C. RSI
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

rsi_btc = calculate_rsi(df[TARGET_ASSET])
rsi_btc.name = 'BTC_RSI'

df_features = pd.concat([df_log, df_vol_recent, rsi_btc], axis=1)

target_hourly = df_log[f'{TARGET_ASSET}_LogRet'].rolling(
    window=FORECAST_HORIZON).std()

target_daily = target_hourly * np.sqrt(24)

target = target_daily.shift(-FORECAST_HORIZON)

dataset_full = df_features.copy()
dataset_full['TARGET_Y'] = target
dataset_full = dataset_full.dropna()

# Salvar referência para GARCH
dataset_full[[f'{TARGET_ASSET}_LogRet']].to_csv("dados_para_garch.csv")

# 4. DIVISÃO TREINO / TESTE
train_size = int(len(dataset_full) * 0.8)
train_data = dataset_full.iloc[:train_size]
test_data = dataset_full.iloc[train_size:]

# 5. NORMALIZAÇÃO (ROBUST SCALER)
scaler = RobustScaler()
feature_cols = [c for c in dataset_full.columns if c != 'TARGET_Y']

# Fit no treino
X_train_scaled = scaler.fit_transform(
    train_data[feature_cols]).astype(np.float32)
# Transform no teste
X_test_scaled = scaler.transform(test_data[feature_cols]).astype(np.float32)

y_train = train_data['TARGET_Y'].values.astype(np.float32)
y_test = test_data['TARGET_Y'].values.astype(np.float32)

df_train_tft = pd.DataFrame(
    X_train_scaled, columns=feature_cols, index=train_data.index)
df_train_tft['y'] = y_train
df_train_tft['tipo'] = 'treino'

df_test_tft = pd.DataFrame(
    X_test_scaled, columns=feature_cols, index=test_data.index)
df_test_tft['y'] = y_test
df_test_tft['tipo'] = 'teste'

# Concatenar
df_tft_final = pd.concat([df_train_tft, df_test_tft])

# Adicionar colunas obrigatórias do NeuralForecast
df_tft_final['ds'] = df_tft_final.index  # Data
df_tft_final['unique_id'] = TARGET_ASSET  # Identificador Único

# Salvar
df_tft_final.to_csv("dados_para_tft.csv")

# Limpeza de memória
del df, df_log, df_vol_recent, df_features, dataset_full, train_data, test_data
gc.collect()

def create_sequences(X, y, time_steps=WINDOW_SIZE):
    num_samples = len(X) - time_steps
    Xs = np.zeros((num_samples, time_steps, X.shape[1]), dtype=np.float32)
    ys = np.zeros((num_samples,), dtype=np.float32)
    for i in range(num_samples):
        Xs[i] = X[i:(i + time_steps)]
        ys[i] = y[i + time_steps]
    return Xs, ys

X_train_final, y_train_final = create_sequences(X_train_scaled, y_train)
X_test_final, y_test_final = create_sequences(X_test_scaled, y_test)

np.save("X_train.npy", X_train_final)
np.save("y_train.npy", y_train_final)
np.save("X_test.npy", X_test_final)
np.save("y_test.npy", y_test_final)

In [ ]:
from arch import arch_model
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_garch = pd.read_csv("dados_para_garch.csv", index_col=0, parse_dates=True)

returns_pct = df_garch.iloc[:, 0] * 100

realized_vol_pct = returns_pct.rolling(window=24).std().shift(-24).dropna()*np.sqrt(24)

n_samples = len(returns_pct)
split_idx = int(n_samples * 0.8)
split_date = returns_pct.index[split_idx]

print(f"Corte Treino/Teste em: {split_date}")

model_garch = arch_model(returns_pct, vol='Garch', p=1, q=1, dist='Normal')
res_garch = model_garch.fit(
last_obs=split_date, disp='off', show_warning=False)

forecasts = res_garch.forecast(

    horizon=24, start=split_date, method='simulation')

pred_var_24h_pct = forecasts.variance.dropna().sum(axis=1)
pred_vol_24h_pct = np.sqrt(pred_var_24h_pct)

common_index = realized_vol_pct.index.intersection(pred_vol_24h_pct.index)

real_vol_decimal = realized_vol_pct.loc[common_index] / 100
pred_garch_decimal = pred_vol_24h_pct.loc[common_index] / 100

# Cálculo do RMSE
if len(real_vol_decimal) == 0:

    print("Interseção de datas vazia.")
else:

    rmse_garch = np.sqrt(mean_squared_error(

        real_vol_decimal, pred_garch_decimal))

    print(f"\n>> GARCH RMSE (Escala Decimal): {rmse_garch:.4f}")


    plt.figure(figsize=(12, 6))


    plt.plot(real_vol_decimal.index, real_vol_decimal,

             color='gray', alpha=0.5, label='Volatilidade Real', linewidth=1)

    plt.plot(pred_garch_decimal.index, pred_garch_decimal,

             color='red', linewidth=1.5, label='Previsão GARCH')

    plt.ylim(0, 0.30)

    # Formatação de Data (Mês-Ano)
    ax = plt.gca()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
    plt.gcf().autofmt_xdate()

    plt.title(

        f"GARCH: Previsão vs Realidade (RMSE: {rmse_garch:.4f})", fontsize=14, fontweight='bold')
    plt.ylabel("Volatilidade (Escala Decimal)", fontsize=12)
    plt.xlabel("Período de Teste", fontsize=12)
    plt.legend(loc='upper left', frameon=True, framealpha=0.9)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()

    plt.show()

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

df_datas = pd.read_csv("dados_para_garch.csv", index_col=0, parse_dates=True)
datas_teste = df_datas.index[-len(y_test):]

model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(
        X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)  # Saída direta: Volatilidade Diária
])

model.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

predictions = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"\n>> LSTM RMSE: {rmse:.4f}")

plt.figure(figsize=(12, 6))

plt.plot(datas_teste, y_test,
         color='gray', alpha=0.5, label='Volatilidade Real (24h)', linewidth=1)

plt.plot(datas_teste, predictions,
         color='blue', linewidth=1.5, label='Previsão LSTM (24h)')

plt.ylim(0, 0.10)

ax = plt.gca()

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))

plt.gcf().autofmt_xdate()

plt.title(f"LSTM: Previsão vs Realidade (RMSE: {rmse:.4f})",
          fontsize=14, fontweight='bold')
plt.ylabel("Volatilidade (Escala Decimal)", fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MSE
from sklearn.metrics import mean_squared_error
import logging
import torch

# Limpar cache da GPU manualmente antes de começar
torch.cuda.empty_cache()

# Silenciar logs desnecessários
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

df = pd.read_csv("dados_para_tft.csv")
df['ds'] = pd.to_datetime(df['ds'])

# Seleção de colunas numéricas (Features)
required_cols = ['ds', 'unique_id', 'y']
ignore = ['tipo', 'Datetime', 'Unnamed: 0'] + required_cols
candidates = [c for c in df.columns if c not in ignore]
feature_cols = [c for c in candidates if pd.api.types.is_numeric_dtype(df[c])]

# DataFrame Limpo
cols_to_keep = required_cols + feature_cols
df_clean = df[cols_to_keep].copy()

# Recuperar tamanho do teste original
test_size_real = len(df[df['tipo'] == 'teste'])

tft = TFT(
    h=24,
    input_size=168,
    hist_exog_list=feature_cols,
    loss=MSE(),
    max_steps=2000,
    val_check_steps=100,
    learning_rate=5e-4,
    hidden_size=64,
    n_head=4,
    dropout=0.1,
    batch_size=16,
    scaler_type='robust'
)

nf = NeuralForecast(models=[tft], freq='h')

# Ajuste matemático para o tamanho do teste
step_size = 24
resto = (test_size_real - 24) % step_size
test_size_ajustado = test_size_real - resto

fcst_df = nf.cross_validation(
    df=df_clean,
    val_size=0,
    test_size=test_size_ajustado,
    n_windows=None,
    step_size=step_size
)

rmse_tft = np.sqrt(mean_squared_error(fcst_df['y'], fcst_df['TFT']))
print(f"\n>> TFT RMSE: {rmse_tft:.4f}")

plt.figure(figsize=(12, 6))
fcst_df = fcst_df.sort_values('ds')

limit = 300
subset = fcst_df.iloc[:limit]

plt.plot(subset['ds'], subset['y'], color='gray', alpha=0.5, label='Real')
plt.plot(subset['ds'], subset['TFT'], color='purple', linewidth=1.5, label='TFT')

plt.title(f"TFT: Resultado Final (RMSE: {rmse_tft:.4f})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.gcf().autofmt_xdate()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

n_linhas = 15

df_tft_align = fcst_df[['ds', 'y', 'TFT']].copy()
df_tft_align.set_index('ds', inplace=True)

df_lstm_align = pd.DataFrame({'LSTM': predictions}, index=datas_teste)
df_garch_align = pd.DataFrame({'GARCH': pred_garch_decimal})

df_erros = df_tft_align.join(df_lstm_align, how='inner').join(df_garch_align, how='inner')
df_erros.rename(columns={'y': 'Real'}, inplace=True)

df_erros['Erro_LSTM'] = np.abs(df_erros['Real'] - df_erros['LSTM'])
df_erros['Erro_TFT'] = np.abs(df_erros['Real'] - df_erros['TFT'])

df_erros['TFT_Venceu'] = (df_erros['Erro_TFT'] < df_erros['Erro_LSTM']).astype(int)

df_erros['Vitorias_TFT_15h'] = df_erros['TFT_Venceu'].rolling(window=n_linhas).sum()

melhor_indice_data = df_erros['Vitorias_TFT_15h'].idxmax()
posicao_final = df_erros.index.get_loc(melhor_indice_data)
posicao_inicial = posicao_final - n_linhas + 1

fatia_ouro = df_erros.iloc[posicao_inicial : posicao_final + 1].copy()

datas = fatia_ouro.index.strftime('%d/%m %H:%M')

df_tabela = pd.DataFrame({
    'Data/Hora': datas.values,
    'Real': fatia_ouro['Real'].values,
    'GARCH': fatia_ouro['GARCH'].values,
    'LSTM': fatia_ouro['LSTM'].values,
    'TFT': fatia_ouro['TFT'].values
})
colunas_valores = ['Real', 'GARCH', 'LSTM', 'TFT']
for col in colunas_valores:
    df_tabela[col] = (df_tabela[col] * 100).apply(lambda x: f"{x:.2f}%")

fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')
ax.axis('tight')

tabela_visual = ax.table(cellText=df_tabela.values,
                         colLabels=df_tabela.columns,
                         loc='center',
                         cellLoc='center')

tabela_visual.auto_set_font_size(False)
tabela_visual.set_fontsize(12)
tabela_visual.scale(1.2, 1.8)

for (row, col), cell in tabela_visual.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#4C72B0')
    elif col == 4:
        cell.set_facecolor('#E8F5E9')
        cell.set_text_props(weight='bold', color='#1B5E20')

plt.title("Comparação Percentual de Previsão",
          fontweight='bold', fontsize=14, pad=20)

plt.tight_layout()

nome_imagem = "tabela_comparativa_visual_tft_vencedor.png"
plt.savefig(nome_imagem, dpi=300, bbox_inches='tight')

vitorias = int(df_erros.loc[melhor_indice_data, 'Vitorias_TFT_15h'])
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_tft_align = fcst_df[['ds', 'y', 'TFT']].copy()
df_tft_align.set_index('ds', inplace=True)

df_lstm_align = pd.DataFrame({'LSTM': predictions}, index=datas_teste)
df_garch_align = pd.DataFrame({'GARCH': pred_garch_decimal})

df_full = df_tft_align.join(df_lstm_align, how='inner').join(df_garch_align, how='inner')
df_full.rename(columns={'y': 'Real'}, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)

modelos = ['GARCH', 'LSTM', 'TFT']
cores = ['#D32F2F', '#1976D2', '#388E3C']

limite_zoom = 0.09

for i, modelo in enumerate(modelos):
    ax = axes[i]

    ax.scatter(df_full['Real'], df_full[modelo], alpha=0.4, color=cores[i], edgecolors='w', s=25, linewidths=0.5)

    ax.plot([0, limite_zoom], [0, limite_zoom], 'k--', lw=2, label='Previsão Perfeita')

    ax.set_xlim(0, limite_zoom)
    ax.set_ylim(0, limite_zoom)

    ax.set_title(f'Modelo: {modelo}', fontsize=16, fontweight='bold')
    ax.set_xlabel('Volatilidade Real', fontsize=12)

    if i == 0:
        ax.set_ylabel('Volatilidade Prevista', fontsize=12)

    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')

plt.suptitle("Análise de Dispersão: Volatilidade Real vs. Prevista",
             fontsize=20, fontweight='bold', y=1.05)

plt.tight_layout()

nome_imagem = "scatter_zoom_modelos.png"
plt.savefig(nome_imagem, dpi=300, bbox_inches='tight')

plt.show()